# 04 — Evaluate the saved full JEV screening runs

**Purpose:** compare the two saved full-dataset model runs with the human reference CSV, retaining the complete labelled cohort and reporting unavailable predictions. `True` means retain for full-text assessment; a false negative is a reference-retain study excluded by a model.

| Item | Source |
|---|---|
| Default data mode | `full` — saved real model runs and prepared human-reference CSV |
| Analysis configuration | `configs/evaluations/analysis_full.yaml` |
| Model run selection | The two full workflow YAMLs referenced by the analysis configuration |
| Inputs | Complete prepared JSONL, human labels and existing prediction files |
| Outputs | Real-data coverage; threshold-based metrics and figures only after an explicit threshold is specified |

This notebook makes **no inference/API calls** and needs no API key. It reads saved artifacts without resuming runs or changing their metadata. `configs/notebooks.yaml` remains a separate two-study demonstration; it does not select inputs for the default full evaluation. Choose `DATA_MODE = "demo"` explicitly to use that teaching example.

## 1. Load the complete cohort and archived model outputs

Use the project's `.venv` kernel and restart it after source changes. Run from the top. In full mode, load the actual prepared states and saved results selected by `analysis_full.yaml`. No sampling occurs.

The loader checks source/cohort checksums, ordered IDs, identical archived screening criteria, requested model identities and output integrity. It preserves the inference runs' original code/environment fingerprints. New evaluation code does not require repeating inference. Scientific-input or saved-output mismatches still block evaluation.

This cell neither loads `.env` nor constructs an API client. Missing run files produce an informative prerequisite error rather than falling back silently to synthetic data.

In [ ]:
try:
    from jev_eval.environment import find_project_root, initialize_notebook
except ModuleNotFoundError:
    raise RuntimeError(
        "Select this project's .venv kernel. From the repository terminal, run "
        "uv sync --locked --group notebooks if dependencies are missing."
    ) from None

from jev_eval.config import PrerequisiteError
from jev_eval.evaluation import load_saved_full_evaluation

DATA_MODE = "full"  # Change explicitly to "demo" for the two invented studies.
ANALYSIS_CONFIG = "configs/evaluations/analysis_full.yaml"

if DATA_MODE == "full":
    full_data = load_saved_full_evaluation(find_project_root(), ANALYSIS_CONFIG)
    project = full_data.plans[0].project
    states = full_data.states
elif DATA_MODE == "demo":
    project = initialize_notebook()
    if project.workflow.execution.mode != "demo":
        raise PrerequisiteError("Demo evaluation requires the synthetic demo workflow.")
else:
    raise PrerequisiteError('DATA_MODE must be "full" or "demo".')

In [ ]:
import pandas as pd

from jev_eval.data import load_jsonl, load_states, validate_states

if DATA_MODE == "full":
    display(
        pd.DataFrame(
            [
                {
                    "model": plan.project.models[0].name,
                    "run_id": result.manifest["run_id"],
                    "status": result.manifest["status"],
                    "source_studies": len(states),
                    "dataset": plan.cohort_metadata["source_file"],
                }
                for plan, result in zip(full_data.plans, full_data.results, strict=True)
            ]
        )
    )
else:
    states = validate_states(load_states(project.evaluation.dataset, project.root)).require_valid()
    predictions = load_jsonl(project.root / project.workflow.evaluation.demo_predictions)
    print("SYNTHETIC DEMONSTRATION: invented data, not JEV performance.")

## 2. Load and validate the human reference decisions

Full mode uses the prepared `reference_labels.csv`, joined by the original `study_id`. IDs and ITEM_ID remain strings. The full workflow's explicit mapping converts inclusion status 0 to false and 1 to true; duplicate/unexpected IDs or unmapped labels block loading. Missing labels and ITEM_ID values are reported, never filled with false.

The labelled count includes **all** studies with human decisions, including studies unavailable for model inference. The eleven textless studies remain in the denominator and are counted separately below. Missing/shared ITEM_ID values do not change the study-ID join. Human labels are used only here for evaluation, not sent to a model.

Demo mode uses the old synthetic Boolean JSONL solely for teaching.

In [ ]:
from jev_eval.evaluation import load_reference_labels

if DATA_MODE == "full":
    reference_labels = full_data.reference_labels
    display(full_data.references.summary())
    print("Human reference CSV:", full_data.references.path)
    print("Reference SHA-256:", full_data.references.sha256)
else:
    reference_labels = load_reference_labels(
        project.root / project.workflow.evaluation.reference_labels,
        states=states,
        question=project.scoring.question,
    )
print(f"Reference-labelled citations: {len(reference_labels)}")

## 3. Inspect coverage and explicitly choose a decision threshold

The real outputs contain probabilities, not Boolean eligibility decisions. `decision_threshold: null` in the analysis YAML leaves binary metrics and confusion matrices disabled. Choose and document the threshold according to the evaluation protocol; retain means **P(true) >= threshold**. The same threshold is applied to both models. Do not select it silently or optimise it on these human labels while presenting the result as an independent evaluation.

Coverage is shown even when no threshold has been selected. Both models remain in the table, with the complete human-labelled denominator. Textless, failed, unknown and missing outcomes stay visible and never become negative predictions.

By default, headline metrics are withheld if any labelled study lacks a usable prediction. With a threshold, confusion counts still describe the available valid pairs. `report_partial_metrics: true` explicitly enables metrics conditional on valid labelled pairs; that is not complete-cohort performance. Missingness and the metric scope are retained in the output.

In [ ]:
from jev_eval.evaluation import evaluate_predictions, evaluate_saved_probabilities
from jev_eval.reporting import comparison_table

evaluation = None
if DATA_MODE == "full":
    print("Full-cohort reference and probability coverage:")
    display(full_data.coverage)
    if full_data.settings.decision_threshold is None:
        summary = full_data.coverage.copy()
        print(
            "No decision threshold is specified. Binary metrics and confusion matrices are disabled."
        )
        print(
            "Set and document decision_threshold in", ANALYSIS_CONFIG, "then rerun from Section 1."
        )
    else:
        evaluation = evaluate_saved_probabilities(full_data)
        summary = comparison_table(evaluation)
        print("Decision threshold:", full_data.settings.decision_threshold)
        print("Threshold note:", full_data.settings.threshold_note)
        display(summary)
else:
    evaluation = evaluate_predictions(
        predictions,
        reference_labels,
        project.scoring,
        models=[model.name for model in project.models],
        states=states,
    )
    summary = comparison_table(evaluation)
    display(summary)

## 4. Interpret performance alongside coverage

- **Sensitivity / recall:** among reference-retain studies with valid paired decisions, the proportion retained by the model.
- **Specificity:** among reference-exclude studies with valid paired decisions, the proportion excluded.
- **Precision / positive predictive value (PPV):** among model-retain decisions in those pairs, the proportion retained by the reference.
- **F1:** balances precision and sensitivity at the chosen threshold.
- **F2:** places greater weight on recall; Section 10 provides F2 and WSS@95 readouts for the saved probability data.
- **Balanced accuracy:** the mean of sensitivity and specificity.

`tn`, `fp`, `fn`, `tp` and `valid` identify the evaluated pairs. Undefined denominators produce unavailable values. Read `labelled`, coverage and the explicit unavailable/error counts before interpreting the metrics. The eleven textless studies are neither inferred nor counted as correct exclusions. Partial metrics require an explicit analysis setting and remain conditional on available predictions.

The original two-study demonstration is only a calculation example. Full mode reports the saved real runs. Uncertainty intervals and independent tests of model differences are not calculated here. Separate exploratory cutoff-selection readouts are provided in Sections 8–9.

## 5. Display threshold-based confusion matrices

Rows are human reference decisions and columns are thresholded model decisions. Counts use only valid labelled pairs, so read them with the coverage table. No matrix is calculated until a threshold is explicitly set. Demo figures are prominently labelled as synthetic.

In [ ]:
from jev_eval.reporting import plot_confusion_matrices

figure = None
if evaluation is not None:
    figure = plot_confusion_matrices(evaluation)
    display(figure)
else:
    print(
        "Configured-analysis matrices await a chosen threshold. Exploratory candidate matrices are in Section 8 below."
    )

## 6. Optional analysis exports — writes files

**File-writing warning:** `ALLOW_WRITES = True` plus `allow_writes: true` in the analysis YAML creates a uniquely named private analysis bundle under its `output_root`. It contains coverage, available threshold-based summaries/figures, threshold/partial-coverage settings, reference/prediction checksums and separate inference/analysis provenance. Existing bundles are not overwritten. This step makes no model requests.

With no threshold, only coverage and provenance are exported. Clear notebook outputs before Git; real row-level tables, human labels, figures and executed notebooks remain private. Demo exports retain the existing separate permission guards.

In [ ]:
from jev_eval.evaluation import export_saved_evaluation
from jev_eval.reporting import export_figure, export_table

ALLOW_WRITES = False
if ALLOW_WRITES and DATA_MODE == "full":
    analysis_export_directory = export_saved_evaluation(
        full_data, evaluation, allow_writes=ALLOW_WRITES
    )
    print("Analysis export:", analysis_export_directory)
elif ALLOW_WRITES:
    export_table(summary, project, "results/evaluation/demo/summary.csv", allow_writes=True)
    export_figure(
        figure, project, "results/evaluation/demo/confusion-matrices.png", allow_writes=True
    )
else:
    print("Exports skipped; no files written.")

## 7. Preserve the analysis record

Clear outputs before committing the notebook. Source/configuration/tests belong in Git; real inputs, saved inference artifacts, human labels, analyses and executed notebooks remain in ignored private locations. The existing commit guards reject saved notebook output.

Inference metadata belongs to the original model runs and is not rewritten by evaluation. The analysis configuration records the threshold and coverage policy; an optional export records its own current environment/code hash and the exact source prediction/reference checksums. No training, fine-tuning, API-key access or paid requests are performed by this notebook.

## 8. Explore cutoffs targeting at least 95% observed recall

This section uses existing probabilities only. It does not modify the configured threshold, change model outputs or make API calls. Candidate thresholds are selected and assessed on these same human labels, so they are **post-hoc, exploratory estimates**, not independently validated performance or a guarantee of 95% recall on future studies.

The primary target uses **all human-retain studies** as its denominator. It gives no model credit for unavailable probabilities, reporting `recall_cohort_lower_bound = scored true positives / all human-retain labels`. This does not assign negative predictions to unavailable studies. `recall_scored` is also shown and can be higher because it excludes unavailable cases from its denominator.

In the current data all eleven textless studies were human-retain. Keep every unavailable record in a separate manual-review/retention queue. The model-only matrix below contains scored pairs only. The scenario `recall_if_unscored_retained` describes the separate policy of routing every unscored study forward; it is not the model's recall and is not applied automatically.

For each model, the package selects the **highest observed cutoff** satisfying the target, using `P(true) >= cutoff` and all tied scores. This maximizes automatic exclusions among feasible thresholds. It uses exact observed values rather than interpolated quantiles; unattainable targets are explicitly reported. Use full-precision cutoffs or round downward—rounding upward can drop tied positives.

Review recall, false negatives, specificity, precision and the resulting review queue together. The queue includes model-retained studies plus all unavailable records. Automatic-exclusion counts include false negatives; they are not a claim of safe exclusions. The threshold/configuration in earlier sections remains unset until you deliberately choose it.

For a paper, report this as exploratory threshold selection and validate a locked cutoff on independent data or a prospectively designed held-out evaluation. The saved scores suffice; no new model requests are needed. See [threshold tuning](https://scikit-learn.org/stable/modules/classification_threshold.html) and [validation guidance](https://scikit-learn.org/stable/modules/cross_validation.html).

In [ ]:
from jev_eval.evaluation import (
    select_recall_thresholds,
    threshold_readouts,
    threshold_tradeoff_curve,
)

TARGET_RECALL = 0.95
if DATA_MODE == "full":
    recall_candidates = select_recall_thresholds(full_data, TARGET_RECALL, basis="all_labelled")
    print("Exploratory highest cutoffs meeting the full-labelled recall lower-bound target:")
    display(recall_candidates.style.format({"threshold": "{:.17g}"}, na_rep="not attainable"))

    scored_only_candidates = select_recall_thresholds(full_data, TARGET_RECALL, basis="scored")
    print(
        "Alternative targeting scored-pair recall only; inspect cohort lower bound and manual-retention scenario separately:"
    )
    display(scored_only_candidates.style.format({"threshold": "{:.17g}"}, na_rep="not attainable"))

    baseline_readouts = threshold_readouts(
        full_data, {plan.project.models[0].name: [0.5] for plan in full_data.plans}
    )
    print("Comparison at the conventional 0.5 cutoff:")
    display(baseline_readouts)

    # Illustrative rounded options, evaluated here without changing analysis YAML.
    rounded_cutoffs = {"jev-9b": [0.33], "jev-27b": [0.45]}
    rounded_readouts = threshold_readouts(full_data, rounded_cutoffs)
    print("Illustrative rounded cutoffs; exploratory options, not adopted thresholds:")
    display(
        rounded_readouts[
            [
                "model",
                "threshold",
                "tp",
                "fn",
                "fp",
                "tn",
                "recall_scored",
                "recall_cohort_lower_bound",
                "specificity_scored",
                "ppv_scored",
                "f2_scored",
                "wss95_scored",
                "wss95_cohort_conservative",
                "review_queue_with_unscored",
                "automatic_exclusions",
            ]
        ].style.format(
            {
                "threshold": "{:.2f}",
                "recall_scored": "{:.2%}",
                "recall_cohort_lower_bound": "{:.2%}",
                "specificity_scored": "{:.2%}",
                "ppv_scored": "{:.2%}",
                "f2_scored": "{:.3f}",
                "wss95_scored": "{:.2%}",
                "wss95_cohort_conservative": "{:.2%}",
            },
            na_rep="not available",
        )
    )

    recall_target_comparison = pd.concat(
        [
            select_recall_thresholds(full_data, target, basis="all_labelled")
            for target in (0.95, 0.97, 0.99)
        ],
        ignore_index=True,
    )
    print("Recall targets and required review queues; unattainable targets remain visible:")
    display(
        recall_target_comparison.style.format({"threshold": "{:.17g}"}, na_rep="not attainable")
    )
else:
    print(
        "Threshold exploration uses the saved full-probability dataset, not the Boolean teaching fixture."
    )

## 9. Visualize the recall/workload tradeoff and exploratory confusion matrices

Solid recall curves use the entire human-retain denominator, with zero credit for unavailable probabilities. Dotted curves use scored human-retain studies only. The dashed vertical lines are the model-specific candidate cutoffs. The exclusion curve shows the fraction removed from the review queue, including incorrectly excluded positives.

The matrices show true/false positives and negatives **among scored pairs only**. Unavailable studies are reported outside each matrix and remain in the separate manual queue. These are explicitly exploratory matrices; they are available even while the earlier configured-analysis threshold remains null. Both models can have different candidate thresholds because this comparison targets equal minimum recall rather than equal numeric probability.

In [ ]:
from jev_eval.reporting import plot_recall_candidate_matrices, plot_threshold_tradeoffs

if DATA_MODE == "full":
    recall_curve = threshold_tradeoff_curve(full_data)
    threshold_tradeoff_figure = plot_threshold_tradeoffs(
        recall_curve, recall_candidates, TARGET_RECALL
    )
    display(threshold_tradeoff_figure)
    exploratory_confusion_figure = plot_recall_candidate_matrices(recall_candidates)
    if exploratory_confusion_figure is not None:
        display(exploratory_confusion_figure)
    else:
        print("The target is unattainable with the available scores; no candidate matrix is shown.")

## 10. Compare specificity, PPV, F2 and work saved at 95% recall

This table reuses the package calculations from Section 8. It compares the highest feasible 95%-target cutoffs, the illustrative rounded cutoffs and the 0.5 baseline. These remain exploratory, measured on the same labels used to select the cutoffs. No API call, file write or change to the configured threshold is made.

**Inputs:** the saved probabilities, human reference labels and cutoffs shown above. **Output:** an in-memory `screening_metric_summary` table with the cutoff source, confusion counts, recall, performance metrics and review workload. Retention uses `P(true) >= cutoff`, including ties.

| Measure | Calculation | Interpretation and scope |
| --- | --- | --- |
| Specificity | `TN / (TN + FP)` | Fraction of human-exclude studies correctly excluded, among scored labelled pairs. |
| Positive predictive value (PPV) | `TP / (TP + FP)` | Fraction of model-retained studies that are human-retain, among scored labelled pairs. Identical to precision. |
| F2 | `5 TP / (5 TP + 4 FN + FP)` | Combines recall and PPV, placing four times the weight on false negatives compared with false positives in this count formula. Uses scored labelled pairs; unavailable studies remain outside the matrix. |
| WSS@95, scored | `(TN + FN) / N_scored_labelled - 0.05` | Work saved over random sampling at the 95% recall target, conditional on scored labelled pairs. Shown only if their observed recall is at least 95%. |
| WSS@95, cohort conservative | `automatic_exclusions / N_all - 0.05`, equivalently `0.95 - review_queue_with_unscored / N_all` | All unscored studies stay in the manual-review queue. Shown only with complete reference labels and at least 95% recall using **all human-retain labels**, giving no credit for unscored positives. |

WSS@95 is a proportion of studies, not money or measured researcher time. A value of 0.70 means 70 percentage points of work saved beyond the 5% saved by random sampling at the 95% target. Negative values are retained. The table uses the actual workload at each displayed cutoff; tied scores can make achieved recall exceed 95%. No interpolation, tie-breaking or manual-review recovery is used to claim model recall. Values at more cautious cutoffs can be lower than those at the highest feasible cutoff.

Read both recall denominators alongside WSS. All eleven currently unscored studies are human-retain. They are not treated as model negatives or credited as model successes. A blank WSS means the target is not met in that scope, labels are incomplete, or the required denominator is unavailable. Other undefined measures also remain unavailable rather than becoming zero.

The WSS definition follows [Cohen et al. (2006)](https://academic.oup.com/jamia/article-pdf/13/2/206/2178574/13-2-206.pdf); see also the [F-beta definition](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.fbeta_score.html).

In [ ]:
if DATA_MODE == "full":
    metric_readouts = pd.concat(
        [
            recall_candidates.assign(cutoff_source="Highest cutoff meeting cohort 95% target"),
            rounded_readouts.assign(cutoff_source="Illustrative rounded cutoff"),
            baseline_readouts.assign(cutoff_source="0.5 baseline"),
        ],
        ignore_index=True,
    )
    screening_metric_summary = metric_readouts.reindex(
        columns=[
            "model",
            "cutoff_source",
            "threshold",
            "scored_pairs",
            "tp",
            "fn",
            "fp",
            "tn",
            "recall_scored",
            "recall_cohort_lower_bound",
            "specificity_scored",
            "ppv_scored",
            "f2_scored",
            "wss95_scored",
            "wss95_cohort_conservative",
            "unscored_studies",
            "review_queue_with_unscored",
        ]
    )
    print("Exploratory screening metrics: scored-pair performance and separate cohort workload")
    display(
        screening_metric_summary.style.format(
            {
                "threshold": "{:.17g}",
                "scored_pairs": "{:.0f}",
                "tp": "{:.0f}",
                "fn": "{:.0f}",
                "fp": "{:.0f}",
                "tn": "{:.0f}",
                "recall_scored": "{:.2%}",
                "recall_cohort_lower_bound": "{:.2%}",
                "specificity_scored": "{:.2%}",
                "ppv_scored": "{:.2%}",
                "f2_scored": "{:.3f}",
                "wss95_scored": "{:.2%}",
                "wss95_cohort_conservative": "{:.2%}",
                "unscored_studies": "{:.0f}",
                "review_queue_with_unscored": "{:.0f}",
            },
            na_rep="not available",
        )
    )
else:
    print(
        "These cutoff readouts require DATA_MODE='full' and saved probabilities; the Boolean demo remains above."
    )